# Retail Orders — Data Profiling Notebook

**Purpose:** Profile the supplied raw retail orders data for completeness, uniqueness, validity, consistency, and freshness. Run all cells from top to bottom.



In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# In GitHub, keep the CSV in the data/ folder as shown in README.
DATA_PATH = Path("data/retail-orders-raw.csv")
df = pd.read_csv(DATA_PATH)
print(f"Rows: {len(df)}, Columns: {len(df.columns)}")
display(df.head())


## 1. Standardize fields for profiling

The raw file is preserved. Normalized values are created in a working copy for checks and KPI calculations.


In [ ]:
profile = df.copy()
profile.columns = profile.columns.str.strip().str.lower()
for col in ["customer_segment", "city", "category", "payment_status"]:
    profile[col] = profile[col].astype("string").str.strip()

profile["customer_segment_norm"] = profile["customer_segment"].str.title()
profile["category_norm"] = profile["category"].str.title()
profile["payment_status_norm"] = profile["payment_status"].str.title()
profile["quantity_num"] = pd.to_numeric(profile["quantity"], errors="coerce")
profile["unit_price_num"] = pd.to_numeric(profile["unit_price"], errors="coerce")
profile["discount_num"] = pd.to_numeric(profile["discount_pct"], errors="coerce")
profile["order_date_parsed"] = pd.to_datetime(profile["order_date"], errors="coerce", dayfirst=True)


## 2. Completeness

Required fields should not be null or blank. Discount is conditionally nullable and must be approved before assuming zero.


In [ ]:
required = ["order_id", "order_date", "customer_segment", "city", "category",
            "quantity", "unit_price", "payment_status"]
completeness = pd.DataFrame({
    "missing_count": profile[required].isna().sum(),
    "missing_pct": (profile[required].isna().mean() * 100).round(2)
})
display(completeness)


## 3. Uniqueness


In [ ]:
duplicate_ids = profile[profile.duplicated("order_id", keep=False)].sort_values("order_id")
print("Duplicate order_id rows:", len(duplicate_ids))
display(duplicate_ids)


## 4. Validity


In [ ]:
validity_checks = {
    "quantity_is_positive_whole_number": profile["quantity_num"].notna() & (profile["quantity_num"] > 0) & (profile["quantity_num"] % 1 == 0),
    "unit_price_is_nonnegative": profile["unit_price_num"].notna() & (profile["unit_price_num"] >= 0),
    "discount_between_0_and_100": profile["discount_num"].notna() & profile["discount_num"].between(0, 100),
    "date_is_parseable": profile["order_date_parsed"].notna(),
    "segment_is_allowed": profile["customer_segment_norm"].isin(["Student", "Fresher", "Professional"]),
    "category_is_allowed": profile["category_norm"].isin(["Learning Kit", "Course Access", "Mentor Session"]),
    "payment_status_is_allowed": profile["payment_status_norm"].isin(["Paid", "Pending", "Failed", "Refunded"]),
}
validity = pd.DataFrame({
    "passed": {k: int(v.sum()) for k, v in validity_checks.items()},
    "failed": {k: int((~v).sum()) for k, v in validity_checks.items()}
})
display(validity)


## 5. Consistency


In [ ]:
# Case/whitespace variants are normalized above; report distinct raw vs normalized values.
consistency = pd.DataFrame({
    "raw_distinct": [profile[c].nunique(dropna=True) for c in ["customer_segment", "category", "payment_status"]],
    "normalized_distinct": [profile[c].nunique(dropna=True) for c in ["customer_segment_norm", "category_norm", "payment_status_norm"]]
}, index=["customer_segment", "category", "payment_status"])
display(consistency)

# Check order-level duplicates and numeric/date coherence.
print("Rows with duplicate order IDs:", int(profile["order_id"].duplicated(keep=False).sum()))


## 6. Freshness

The dictionary expects order dates from 2025-01-01 through today. This check uses the runtime date when the notebook is executed.


In [ ]:
today = pd.Timestamp.today().normalize()
min_allowed = pd.Timestamp("2025-01-01")
freshness = pd.DataFrame({
    "check": ["date_present_and_parseable", "date_not_before_2025_01_01", "date_not_in_future"],
    "failed_rows": [
        int(profile["order_date_parsed"].isna().sum()),
        int((profile["order_date_parsed"] < min_allowed).fillna(False).sum()),
        int((profile["order_date_parsed"] > today).fillna(False).sum())
    ]
})
display(freshness)


## 7. Summary and initial observations


In [ ]:
print("PROFILE SUMMARY")
print(f"Rows inspected: {len(profile)}")
print(f"Duplicate order IDs: {profile['order_id'].duplicated(keep=False).sum()}")
print(f"Missing order dates: {profile['order_date'].isna().sum()}")
print(f"Missing cities: {profile['city'].isna().sum()}")
print(f"Unparseable dates: {profile['order_date_parsed'].isna().sum()}")
print(f"Invalid quantities: {(~validity_checks['quantity_is_positive_whole_number']).sum()}")
print(f"Invalid discounts: {(~validity_checks['discount_between_0_and_100']).sum()}")
print("\nDo not overwrite the raw source. Resolve issues with the data owner and retain an audit trail.")
